# Operadores de Kraus y Baños Térmicos (Ruido Qiskit)

Las compuertas cuánticas ideales ya son historia. Ahora vamos a emular mediante Matrices Formadoras Kraus ($K_i$) cómo la radiación de calor y defase destruye y gotea asintóticamente la vida de nuestros Qubits (Canales Asimétricos C.P.T.P).

In [ ]:
from qiskit_aer.noise import NoiseModel, pauli_error, amplitude_damping_error
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
import numpy as np

## 1. El canal Amplitude Damping ($T_1$ Relaxation)
Decaimiento o pérdida de energía hacia el estado tierra $|0\rangle$. Si creamos un estado excitado mágico en las alturas $|1\rangle$ y esperamos lo suficiente, la física de Kraus empujará al cubo disipativamente a relajarse.

In [ ]:
# Parametro de Probabilidad Gamma (Simulando un goteo muy severo del 30%)
fall_prob = 0.3 
error_t1 = amplitude_damping_error(fall_prob)

print("Visualizando Operadores Fundamentales de Kraus K_0 y K_1 del Canal:")
print("K0 (Permanencia):\n", np.round(error_t1.to_quantumchannel().data[0], 2))
print("\nK1 (Decaimiento a 0):\n", np.round(error_t1.to_quantumchannel().data[1], 2))

## 2. Inyectar el canal ruidoso Kraus a todo el chip 
Instalamos nuestro NoiseModel diabólico que manchará de Damping todas las puertas compuertas `x` del chip virtual.

In [ ]:
from qiskit_aer.primitives import Sampler as AerSampler

noise_model = NoiseModel()
# Añadimos este malvado ruido explícitamente unido a las llamadas computacionales de 'x' gate.
noise_model.add_all_qubit_quantum_error(error_t1, ['x']) 

print(noise_model)

## 3. Demostración Empírica Práctica
Sin Ruido, 1000 iteraciones de X deberían darnos estrictamente la cadena final pura `'1'`. Veamos el efecto termodinámico de Lindblad subyacente:

In [ ]:
qc = QuantumCircuit(1)
qc.x(0) # Esta es la parte donde inyectamos la excitacion.
qc.measure_all()

# Simulacion Ideal
ideal_sampler = StatevectorSampler()
res_ideal = ideal_sampler.run([qc], shots=1000).result()[0].data.meas.get_counts()

# Simulacion Afectada por los Agentes Dinámicos (T1)
noisy_sampler = AerSampler(run_options={"noise_model": noise_model})
# En Qiskit Aer Sampler (que es V1), se hace run instanciado
res_noisy = noisy_sampler.run([qc], shots=1000).result().quasi_dists[0]
res_noisy_counts = { '0': int(res_noisy.get(0, 0)*1000), '1': int(res_noisy.get(1, 0)*1000) }

print(f"Laboratorio Universo Perfecto Matemático: {res_ideal}")
print(f"Laboratorio Sujeto a Fugas T1 Relajadas: {res_noisy_counts}")
print("\n(Tristemente, el 30% de naves cuánticas perdieron energía disipativamente fallándonos en el proceso devolviendo ceros fantasma).")